<font size="8">**Modulo 1.1 — Dallo *script* allo *strumento***</font><br>

> Corso Python per sistemisti · Giorno 1 · monitoraggio, affidabilità e troubleshooting


> (c) 2026 Antonio Michele Piemontese




Finora avete usato Python per **scripting**: probabilmnete programmi brevi, lineari, scritti in fretta per
risolvere un compito puntuale. Funzionano — ma uno script di monitoraggio che gira in
produzione viene **rieseguito** ogni notte, **modificato** a mesi di distanza, **condiviso**
nel team e applicato a **centinaia di macchine**. In quel contesto "funziona sul mio caso"
non basta più: serve uno *strumento* affidabile, leggibile e riproducibile.

Questo modulo prende uno script realistico (speriamo simile al vostro!) e lo trasforma. Il filo conduttore — che giustifica
ogni passaggio — è **uno solo**, e lo ritroverete in tutto il corso:

1. **Separare la logica dall'I/O.** Il *calcolo* sta in funzioni **pure** (stesso input → stesso
   output, nessun effetto collaterale); leggere file, interrogare DB e agire sui sistemi sta
   ai **bordi**. La logica isolata è leggibile e, soprattutto, **testabile**.
2. **Gestire gli errori in modo esplicito.** Tutto ciò che tocca file, rete, DB o sistemi
   *fallisce*: una riga malformata, un host irraggiungibile, un timeout. Lo strumento deve
   **sopravvivere** e dirti *cosa* è andato storto, non interrompersi in silenzio.
3. **Rendere tutto riproducibile.** Niente percorsi e soglie cablati nel codice: parametri,
   configurazione, e un punto di ingresso pulito che si integra con cron/scheduler.

Seguiamo lo schema **as-is → strumento**: prima la versione com'è scritta oggi, poi la stessa
attività ristrutturata, un *concern* alla volta.

# Rilevamento ambiente di sviluppo

In [ ]:
# impostazione del TOGGLE BINARIO:
try:
    import google.colab                      # package disponibile SOLO in Google Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Running on Colab:", IN_COLAB)


# IMPORT dei package necessari (necessari sia in JN che in Colab):
from IPython.display import Image, display   # import dei package di incorporamento e visualizzazione immagine (una tantum)
                                             # Image e display sono entrambi necessari a Jupyter Notebook
                                             # Google Colab utilizza solo Image
import os                                    # necessario a Google Colab per vedere da una cella codice
                                             # i contenuti del 'content'

Running on Colab: True


# Immagini

Il notebook incorpora le seguenti immagini *png* (in ordine di incorporamento):



# Legenda delle icone (standard)
Legenda delle icone (standard) usate nel notebook (per garantire consistenza semantica):

👉 punto di attenzione, il "succo"<br>
📌 nota / inizio di una nota<br>
📦 punto elenco importante<br>
📊 dati/numeri<br>
🔹 punto elenco normale<br>
⭐ punto elenco importante<br>
✅ punto risolto, positivo<br>
❌ punto negativo, da evitare<br>
⚠️ attenzione, warning, allarme<br>
💡 idea chiave<br>
🧠 idea intuitiva<br>
📝 sintesi / bottom-line<br>
⟶ conseguenza, effetto, passo successivo

# Lo scenario

Avete un gruppo di server applicativi che scrivono un **log di accesso**. Ogni riga è una richiesta, in formato `chiave=valore` (lo stesso stile dei campi di [Splunk](https://www.splunk.com/)):

```
2025-06-20T14:00:03 host=web01 status=200 rt_ms=47
```

- `host` — il server che ha servito la richiesta
- `status` — il codice HTTP (un `5xx` è un errore lato server)
- `rt_ms` — il *response time* in millisecondi

---
> 📌In questo contesto "richiesta" è una **richiesta HTTP**: un client (un browser, un'app, un altro servizio) chiede qualcosa a un server applicativo, e il server risponde. È l'unità elementare del traffico web.
>
>Ogni volta che accade, il server scrive **una riga** nel log di accesso per registrare quella singola interazione. Quindi nel notebook vale l'equazione: *una riga di log = una richiesta servita*. Se `web01` ha 200 righe, ha gestito 200 richieste.
>
> Guardando i campi che usiamo, ogni riga descrive **l'esito** di quella interazione:
>
>- `host=web01` — quale server ha gestito la richiesta
>- `status=200` — com'è andata, col codice di stato HTTP della risposta (200 = OK, 500 = errore del server, ecc.)
>- `rt_ms=47` — quanto è durata, il *response time* in millisecondi (il tempo che il server ha impiegato a rispondere)
>
> <u>Un esempio concreto</u>: un utente apre una pagina web. Il browser manda una richiesta `GET /home` a `web01`; il server la elabora in 47 ms e risponde "200 OK". Il server annota l'accaduto come:<br>
> `... host=web01 status=200 rt_ms=47`.<br>
> Quella riga *è* la richiesta, dal punto di vista del log.
>
> Una singola pagina web "pesante" può generare *decine* di richieste (l'HTML, poi ogni immagine, foglio di stile, script, chiamata API...), ciascuna con la sua riga. Per questo i log di accesso crescono in fretta — ed è esattamente da lì che nascono le metriche di affidabilità del modulo: contando le righe ottieniamo il **volume di richieste**, contando quelle con `status` 5xx ottieniamo l'**error rate**, e guardando la distribuzione di `rt_ms` ottieniamo la **latenza** (inclusa la p95). Tre KPI che escono tutti dalla stessa, semplice unità: la richiesta.
>
> Una precisazione: nel notebook le richieste sono **simulate** dalla funzione `genera_righe` per avere dati su cui lavorare, ma un vero server web (nginx, Apache, un application server Java...) produce log dello stesso tipo **in modo nativo**, una riga per ogni richiesta ricevuta.

---

**Obiettivo del tool:** leggere tutti i log, e per ciascun host calcolare **numero di richieste**,
**error rate** (quota di `5xx`) e **latenza p95** (il 95° percentile dei tempi di risposta).
Poi segnalare gli host che superano le soglie — che è il pane quotidiano del monitoraggio di
prestazioni e affidabilità.

> Usiamo il **p95** e non la media perché la media nasconde le code: è il 5% di richieste lente
> a far arrabbiare gli utenti, e la media le annega. Su questo torneremo nel Giorno 2.

## I dati di esempio

Generiamo due cartelle, per raccontare uno scenario che, come sistemisti, conoscete bene:

- **`logs/`** — i log "puliti" su cui lo script gira in **test**: tutto regolare.
- **`logs_reali/`** — gli stessi dati più la **realtà della produzione**: qualche riga
  malformata (campo mancante, valore non numerico, riga spazzatura) e un file estraneo
  finito nella cartella.

`web02` è volutamente il server **sofferente** (più errori, più code di latenza).

In [ ]:
import random
import shutil
from pathlib import Path
from datetime import datetime, timedelta

In [ ]:
random.seed(42)  # seme fisso => dati riproducibili a ogni esecuzione

In [ ]:
# Profilo di comportamento per ciascun host simulato (3): probabilità di errore, fascia di
# latenza "normale", fascia di latenza "lenta" e quanto spesso capita una richiesta lenta.
HOST_PROFILI = {
    "web01": {"err_prob": 0.01, "rt_base": (30, 90),  "rt_lento": (200, 500),  "prob_lento": 0.02},
    "web02": {"err_prob": 0.12, "rt_base": (80, 220), "rt_lento": (900, 2500), "prob_lento": 0.15},  # malato
    "web03": {"err_prob": 0.02, "rt_base": (40, 110), "rt_lento": (250, 600),  "prob_lento": 0.03},
}

HOST_PROFILI

{'web01': {'err_prob': 0.01,
  'rt_base': (30, 90),
  'rt_lento': (200, 500),
  'prob_lento': 0.02},
 'web02': {'err_prob': 0.12,
  'rt_base': (80, 220),
  'rt_lento': (900, 2500),
  'prob_lento': 0.15},
 'web03': {'err_prob': 0.02,
  'rt_base': (40, 110),
  'rt_lento': (250, 600),
  'prob_lento': 0.03}}

**Cosa è `HOST_PROFILI`?**<br>
E' un **dizionario** che definisce il **profilo di comportamento di ciascun host simulato**: descrive *come* si comporta ogni server, così da poter fabbricare righe di log sintetiche ma realistiche.

La struttura è `nome_host → dizionario di parametri`, con quattro chiavi per host:

- `err_prob` — la probabilità che una richiesta sia un errore `5xx` (es. `0.12` = circa il 12% di errori)
- `rt_base` — la coppia `(min, max)` della latenza "normale" in ms
- `rt_lento` — la coppia `(min, max)` della latenza "della coda lenta" in ms
- `prob_lento` — quanto spesso una richiesta finisce nella fascia lenta invece che in quella normale

> ---
> **📌 Il processo generativo dei dati simulati**<br>
> `HOST_PROFILI` *è* la specifica del **processo generativo** dei dati simulati. Contiene i "veri" parametri del modello — `err_prob`, `prob_lento`, le fasce di latenza — da cui `genera_righe` **campiona** per produrre ogni riga.
>
> Detto nel linguaggio probabilistico-statistico, è **un modello generativo parametrico** molto semplice. Ogni riga di log è un campione estratto da quelle distribuzioni:
> - lo `status` viene da una Bernoulli (errore sì/no con probabilità `err_prob`) seguita da una categoriale sui codici,
> - la latenza `rt_ms` viene da una **mixture** di due uniformi — il regime "normale" `rt_base` e la "coda lenta" `rt_lento` — pesate da `prob_lento`. È, in piccolo, lo stesso schema di una mixture distribution.
>
> E qui c'è il dettaglio che rende l'esempio didatticamente onesto: quei parametri sono i **parametri veri ma latenti**. Lo strumento di monitoraggio che costruiamo nel notebook **non li conosce** — vede solo i campioni (le righe di log) e cerca di *risalire* alle proprietà del sistema stimando *error rate* e *p95* dai dati osservati. È esattamente **la situazione del troubleshooting reale**: i sistemisti non hanno accesso al "vero" `err_prob` di un server, **lo inferiscono dalle metriche**. Noi, come autori del notebook, conosciamo `HOST_PROFILI`: questo ci dà la *ground truth* contro cui verificare che lo strumento funzioni: sappiamo che `web02` è davvero malato (`err_prob: 0.12`), quindi se lo strumento lo segnala come critico sta facendo il suo lavoro.
>
> Un paio di precisazioni per inquadrarlo bene.
> - il `random.seed` fissato rende il processo generativo *riproducibile*: stessa seme, stessa realizzazione del campione — comodo per la didattica, ma è un campione particolare, non l'unico possibile.
> - il limite del modello: è deliberatamente ingenuo (campioni i.i.d., nessuna correlazione temporale, nessuna autocorrelazione, nessun pattern giorno/notte) perché il suo scopo è solo dare "materia prima" (dati di produzione) plausibile allo strumento, non modellare fedelmente il traffico reale. Più avanti, nel modulo 1.3, introdurremo tempo e correlazione.
>
> ---



---
La seguente funzione `genera_righe(host, profilo, n, t0)` legge questi parametri e per ogni riga "tira i dadi": con probabilità `err_prob` produce uno status `5xx`, e con probabilità `prob_lento` pesca la latenza da `rt_lento` anziché da `rt_base`. È per questo che `web02` ha valori volutamente più alti (`err_prob: 0.12`, `prob_lento: 0.15`, coda fino a 2500 ms): è il server **"malato"**, tarato perché lo strumento di monitoraggio lo segnali come critico su entrambe le metriche, mentre `web01` e `web03` restano sani.

`HOST_PROFILI` **non fa parte dello strumento** che stiamo insegnando a costruire. Serve solo a generare i file di log di esempio, per rendere il notebook autosufficiente e riproducibile (seme fisso → stessi dati a ogni esecuzione). Lo strumento vero — `parse_riga`, `leggi_eventi`, `aggrega`, … — non vede mai `HOST_PROFILI`: vede soltanto i log che ne risultano, esattamente come in produzione vedrebbe i log veri dei server.

In [ ]:
def genera_righe(host, profilo, n, t0):
    """Genera n righe di log sintetiche ma realistiche per UN host.

    NB: questa funzione serve solo a fabbricare i dati di esempio, così il
    notebook è autosufficiente e riproducibile. NON fa parte dello strumento
    che stiamo costruendo: lo strumento vedrà soltanto i file di log prodotti,
    esattamente come in produzione vedrebbe i log veri dei server.

    Parametri
    ---------
    host    : str   -> nome del server, finirà nel campo `host=` di ogni riga
    profilo : dict  -> i parametri di comportamento dell'host (da HOST_PROFILI):
                       err_prob, rt_base, rt_lento, prob_lento
    n       : int   -> quante righe di log generare
    t0      : datetime -> istante di partenza del primo log

    Ritorna
    -------
    list[str] -> le n righe di log, ciascuna nel formato `chiave=valore`.
    """
    righe = []          # accumulatore: qui mettiamo le righe man mano che le creiamo
    t = t0              # "orologio" interno; partiamo dall'istante iniziale.
                        # datetime è IMMUTABILE: t += ... crea un NUOVO datetime e lo
                        # riassegna a t, senza toccare t0. Così ogni host parte dallo
                        # stesso istante (t0 resta intatto per il prossimo host).

    for _ in range(n):  # ripetiamo n volte. `_` è la convenzione per "indice che non uso":
                        # ci interessa solo eseguire il corpo n volte, non il contatore.

        # --- 1) TIMESTAMP che avanza ---
        # Aggiungiamo da 1 a 4 secondi a ogni iterazione, per simulare richieste
        # ravvicinate ma con orari crescenti (un log reale non ha due righe identiche).
        t += timedelta(seconds=random.randint(1, 4))
        ts = t.strftime("%Y-%m-%dT%H:%M:%S")   # formato ISO 8601, es. "2025-06-20T14:00:03"

        # --- 2) CODICE DI STATO HTTP ---
        # random.random() estrae un float in [0.0, 1.0). Confrontarlo con una soglia
        # equivale a "tirare i dadi": è vero, in media, err_prob volte su 1.
        if random.random() < profilo["err_prob"]:
            # caso ERRORE: un 5xx (errore lato server) a caso fra questi tre
            status = random.choice([500, 502, 503])
        else:
            # caso OK: un 2xx/3xx. Il 200 è ripetuto 3 volte su 5 di proposito:
            # ripetere un valore nella lista è un modo semplice per "pesarlo"
            # (qui ~60% 200, 20% 201, 20% 304). Alternativa più esplicita:
            #   random.choices([200, 201, 304], weights=[6, 2, 2])
            status = random.choice([200, 200, 200, 201, 304])

        # --- 3) LATENZA (response time in ms) ---
        # Ogni tanto (prob_lento) una richiesta finisce nella "coda lenta", per
        # simulare i picchi di latenza che un buon monitoraggio deve scovare.
        if random.random() < profilo["prob_lento"]:
            # `*` spacchetta la tupla (min, max): se rt_lento = (900, 2500),
            # questo diventa random.randint(900, 2500).
            rt = random.randint(*profilo["rt_lento"])
        else:
            rt = random.randint(*profilo["rt_base"])   # latenza "normale"

        # --- 4) COMPOSIZIONE della riga ---
        # f-string: inseriamo i valori nel formato `chiave=valore` separato da spazi,
        # cioè lo STESSO formato che parse_riga() dovrà poi interpretare.
        righe.append(f"{ts} host={host} status={status} rt_ms={rt}")

    return righe

La seguente cella crea la cartella PULITA (lo scenario di TEST: dati senza imperfezioni).

---
**Alcune anticipazioni**<br>
`Path` è la **classe** per rappresentare percorsi di file e cartelle in modo orientato agli oggetti. Viene dal modulo `pathlib` della libreria standard (la famosa [***standard library di Python***](https://realpython.com/ref/best-practices/stdlib/?utm_source=notification_summary&utm_medium=email&utm_campaign=2026-06-22), ed è il modo moderno di lavorare col filesystem in Python — al posto di manipolare i percorsi come semplici stringhe con il vecchio modulo `os.path`.

L'idea di fondo: invece di trattare "logs/web01.log" come una stringa qualunque, lo incapsuliamo in un oggetto `Path` che "sa" di essere un percorso e ci mette a disposizione metodi e operatori pensati apposta. Quando scriviamo `Path("logs")` non succede nulla sul disco: stiamo solo creando l'oggetto che rappresenta quel percorso. È un'azione separata dall'eventuale toccare il disco (creare, leggere, scrivere), che avviene solo quando chiamiamo i metodi opportuni.

Compone i percorsi con l'operatore `/`. Invece di concatenare stringhe a mano (rischiando di sbagliare gli slash), scriviamo `Path("logs") / "web01.log"`. Tra l'altro `pathlib` usa il separatore giusto per il sistema operativo: "/" su Linux/Mac, "\" su Windows, quindi lo stesso codice funziona ovunque senza che ce ne preoccupiamo. Porta con sé i metodi sul filesystem. L'oggetto `Path` espone direttamente le operazioni, che nei notebook compaiono così:

`Path("logs").mkdir(exist_ok=True)` — crea la cartella<br>
`percorso.write_text(testo, encoding="utf-8")` — scrive un file in una sola chiamata<br>
`percorso.read_text(encoding="utf-8")` — lo rilegge<br>
`cartella.glob("*.log")` — elenca i file che corrispondono a un pattern<br>
`cartella.iterdir()` — scorre il contenuto di una cartella<br>
`percorso.open(...)` — apre il file (come `open()`, ma a partire dall'oggetto)

Ci sono poi gli attributi che danno i pezzi del percorso senza parsing manuale:
- `percorso.name` (il nome del file, *web01.log*),
- `percorso.stem` (senza estensione, *web01*),
- `percorso.suffix` (l'estensione, *.log*),
- `percorso.parent` (la cartella che lo contiene).

Nel notebook 1.2, per esempio, la funzione `apri_testo` userà un proprio `percorso.suffix == ".gz"` per decidere se aprire il file con *gzip* — molto più pulito che cercare *".gz"* in una stringa.

Il contrasto con il vecchio approccio rende l'idea. Prima si scriveva:

In [ ]:
import os
percorso = os.path.join("logs", "web01.log")   # comporre
os.makedirs("logs", exist_ok=True)             # creare
nome = os.path.basename(percorso)              # estrarre il nome

Con `pathlib` diventa più leggibile e tutto "ruota" attorno all'oggetto:

In [ ]:
from pathlib import Path
percorso = Path("logs") / "web01.log"
Path("logs").mkdir(exist_ok=True)
nome = percorso.name

In sintesi: `Path` ci dà un **oggetto-percorso che compone i cammini in modo portabile** e ci offre, come **metodi*é, tutte le **operazioni su file e cartelle** — il che rende il codice <u>più leggibile e meno soggetto a errori</u> rispetto alle stringhe grezze. È esattamente per questo che nei notebook useremo `Path(...)` ovunque invece di passare nomi di file come semplici stringhe.

---

Ora possiamo davvero creare la cartella pulita:

In [ ]:
# ============================================================================
# CARTELLA PULITA — lo scenario di TEST: log "perfetti", senza imperfezioni.
# Generiamo un file di log per ciascun host (come in produzione, dove spesso
# ogni server/servizio scrive il proprio file, che poi un collettore aggrega).
# ============================================================================

# Path("logs") crea un OGGETTO percorso (pathlib): NON tocca ancora il disco, è
# solo una rappresentazione del percorso "logs". pathlib è il modo moderno di
# gestire file e cartelle, più pulito delle stringhe + os.path.
# .mkdir() crea fisicamente la cartella sul disco.
#   exist_ok=True -> se la cartella esiste già NON solleva un errore (utile perché
#                    il notebook può essere rieseguito più volte; senza, la seconda
#                    esecuzione fallirebbe con FileExistsError).
#   (per creare anche le eventuali cartelle padre mancanti si userebbe parents=True)
Path("logs").mkdir(exist_ok=True)

# datetime(anno, mese, giorno, ora, minuto, secondo): costruisce l'istante iniziale.
# È lo STESSO t0 per tutti gli host, quindi tutti i log partono dal medesimo momento
# (ricorda: genera_righe non modifica t0, perché datetime è immutabile).
t0 = datetime(2025, 6, 20, 14, 0, 0)

# Iteriamo il dizionario dei profili. .items() restituisce coppie (chiave, valore),
# che spacchettiamo al volo in due variabili:
#   host -> il nome del server (la CHIAVE, una stringa: "web01", "web02", ...)
#   prof -> il suo profilo di comportamento (il VALORE, il dict dei parametri)
# (Iterare il dict "nudo", senza .items(), darebbe le sole chiavi.)
for host, prof in HOST_PROFILI.items():

    # Generiamo 200 righe di log per QUESTO host, secondo il suo profilo.
    # genera_righe restituisce una list[str]: una stringa per ogni riga di log.
    righe = genera_righe(host, prof, 200, t0)

    # --- Scrittura del file ---
    # f"logs/{host}.log" costruisce il nome del file, es. "logs/web01.log".
    #   (Alternativa più idiomatica con pathlib:  Path("logs") / f"{host}.log")
    #
    # "\n".join(righe) unisce le righe inserendo un a-capo TRA una e l'altra:
    # per N righe mette N-1 a-capo, quindi l'ULTIMA riga resterebbe senza.
    # Il + "\n" finale aggiunge l'a-capo conclusivo, così OGNI riga (ultima
    # inclusa) termina correttamente -> file di testo "ben formato".
    #
    # .write_text(...) è la scorciatoia di pathlib che APRE il file, scrive
    # l'intera stringa e lo RICHIUDE, tutto in un'unica chiamata.
    #   encoding="utf-8" SEMPRE esplicito: non affidarsi al default del sistema
    #   operativo (su Windows può essere cp1252) per evitare sorprese tra ambienti.
    Path(f"logs/{host}.log").write_text("\n".join(righe) + "\n", encoding="utf-8")

Due punti:
* il perché del + "\n" finale: il join mette i separatori tra gli elementi, quindi l'ultima riga resterebbe senza terminazione
* `encoding="utf-8"` esplicito non è pedanteria — su Windows il default può essere `cp1252`, e affidarvisi è proprio la causa dei bug di encoding che vedremo nel modulo 1.2.

L'alternativa: `pathlib Path("logs") / f"{host}.log"`, che è il modo più idiomatico di comporre i percorsi.

Ora creiamo la cartella reale (di produzione) e poi facciamo alcune note:

In [ ]:
# 2) Cartella REALE (la PRODUZIONE: dati puliti + imperfezioni) ---------------------
# rmtree con ignore_errors=True: partiamo sempre da una cartella pulita, anche se
# il notebook viene rieseguito più volte (operazione idempotente, niente errori).
shutil.rmtree("logs_reali", ignore_errors=True)
Path("logs_reali").mkdir()
for p in Path("logs").glob("*.log"):           # glob("*.log") -> solo i file di log...
    shutil.copy(p, Path("logs_reali") / p.name)  # ...copiati così come sono nella cartella "reale"

Alcune note per la comprensione della cella precedente:

- **`shutil`** è il modulo della libreria standard per operazioni "di alto livello" su file e cartelle (copiare, spostare, eliminare interi alberi). `pathlib` gestisce un percorso alla volta; `shutil` lavora su intere strutture.

- **`shutil.rmtree`** cancella una cartella **e tutto il suo contenuto in modo ricorsivo** (sottocartelle e file inclusi). È un'operazione **distruttiva e definitiva** — niente cestino — quindi va usata con attenzione sul percorso giusto. Nota in più: `ignore_errors=True` copre *anche* il caso della **prima** esecuzione, quando `logs_reali` non esiste ancora (senza, mancando la cartella, solleverebbe un errore); così funziona sia al primo giro sia ai successivi.

- Il **`mkdir()` qui è senza `exist_ok=True`** — al contrario della cartella `logs`. È voluto: `rmtree` ha appena garantito che la cartella non c'è più, quindi la ricreiamo pulita; se per qualche motivo esistesse ancora, `mkdir` solleverebbe un errore, fungendo da piccolo controllo di sanità.

- **`glob` restituisce un iteratore "pigro" di oggetti `Path`** (non una lista): valuta i file man mano. Per questo `p` è già un `Path`, e puoi usarne `p.name` e passarlo direttamente a `shutil.copy`. Il `*` è un *wildcard* (qualsiasi sequenza di caratteri) e `glob` cerca **solo nella cartella indicata**, non nelle sottocartelle (per quello servirebbe `rglob`).

- **`shutil.copy(sorgente, destinazione)`** copia il contenuto del file **e i bit di permesso** (non i timestamp: per quelli c'è `copy2`; per il solo contenuto `copyfile`). Qui la destinazione `Path("logs_reali") / p.name` riusa lo **stesso nome del file**, così `logs/web01.log` diventa `logs_reali/web01.log`.

> **L'intento d'insieme di questa cella**: la cartella "reale" si costruisce partendo da una **copia dei log puliti** come base, su cui poi (nella cella successiva) vengono *aggiunte* le righe sporche. I dati puliti restano la base, le imperfezioni sono uno strato sopra — mantenendo la riproducibilità.

In [ ]:
# ...e ora iniettiamo la "realtà": righe che il parser INGENUO non si aspetta.
# Ognuna rappresenta un caso di sporcizia tipico dei log veri:
righe_sporche = [
    "2025-06-20T15:01:10 host=web02 status=500",            # manca il campo rt_ms
    "2025-06-20T15:01:11 host=web02 status=200 rt_ms=NaN",  # rt_ms presente ma non numerico
    "QUESTA RIGA E' ROTTA",                                 # nessun campo chiave=valore
    "2025-06-20T15:01:13 status=200 rt_ms=55",              # manca il campo host
]
# Modalità "a" = APPEND: aggiunge in coda a web02.log SENZA sovrascrivere i dati
# puliti appena copiati (la modalità "w" li cancellerebbe).
with open("logs_reali/web02.log", "a", encoding="utf-8") as f:
    for r in righe_sporche:
        f.write(r + "\n")

# Un file ESTRANEO finito per errore nella cartella dei log: serve a verificare che
# lo strumento legga solo i veri file di log e ignori il resto (lo vedremo con glob).
Path("logs_reali/README.txt").write_text("Cartella dei log applicativi.\n", encoding="utf-8")

print("Creati:", [p.name for p in sorted(Path("logs").iterdir())],
      "in logs/ e", [p.name for p in sorted(Path("logs_reali").iterdir())], "in logs_reali/")


Creati: ['web01.log', 'web02.log', 'web03.log'] in logs/ e ['README.txt', 'web01.log', 'web02.log', 'web03.log'] in logs_reali/


Diamo ora un'occhiata al formato (dati puliti) e alla coda "sporca" della produzione che abbiamo creato nelle celle precedenti:

In [ ]:
print("--- logs/web02.log (prime 5 righe, dati puliti) ---")
print("\n".join(Path("logs/web02.log").read_text(encoding="utf-8").splitlines()[:5]))

print("\n--- logs_reali/web02.log (ultime 6 righe: 2 pulite + 4 sporche) ---")
print("\n".join(Path("logs_reali/web02.log").read_text(encoding="utf-8").splitlines()[-6:]))


--- logs/web02.log (prime 5 righe, dati puliti) ---
2025-06-20T14:00:02 host=web02 status=201 rt_ms=126
2025-06-20T14:00:04 host=web02 status=304 rt_ms=141
2025-06-20T14:00:08 host=web02 status=200 rt_ms=145
2025-06-20T14:00:12 host=web02 status=200 rt_ms=199
2025-06-20T14:00:15 host=web02 status=200 rt_ms=1607

--- logs_reali/web02.log (ultime 6 righe: 2 pulite + 4 sporche) ---
2025-06-20T14:08:39 host=web02 status=503 rt_ms=142
2025-06-20T14:08:40 host=web02 status=304 rt_ms=139
2025-06-20T15:01:10 host=web02 status=500
2025-06-20T15:01:11 host=web02 status=200 rt_ms=NaN
QUESTA RIGA E' ROTTA
2025-06-20T15:01:13 status=200 rt_ms=55


## 1. La versione AS-IS

Ecco come potrebbe essere scritto il vostro script oggi: **un unico blocco lineare**. Apre i file, spezza le righe,
accumula i conteggi in **dizionari paralleli** (`richieste`, `errori`, `latenze`), calcola i
KPI e stampa il report — tutto mescolato. Le soglie sono **numeri magici** sparsi nel codice,
i percorsi sono **cablati**.

È il modo in cui uno script *nasce*. Lo facciamo girare sui dati di **test** (`logs/`).

In [ ]:
import os

CARTELLA = "logs"   # <-- percorso cablato nel codice

richieste = {}      # host -> numero di richieste
errori = {}         # host -> numero di 5xx          (dizionari "paralleli": fragile)
latenze = {}        # host -> lista dei rt, per il p95

for nome_file in os.listdir(CARTELLA):                  # legge TUTTO ciò che trova nella cartella
    percorso = os.path.join(CARTELLA, nome_file)
    f = open(percorso)                                  # niente context manager, niente encoding
    for riga in f:
        parti = riga.split()
        host = None
        status = None
        rt = None
        for token in parti[1:]:                         # parti[0] è il timestamp
            chiave, valore = token.split("=")           # assume sempre esattamente un '='
            if chiave == "host":
                host = valore
            elif chiave == "status":
                status = int(valore)                    # assume sempre numerico
            elif chiave == "rt_ms":
                rt = int(valore)
        if host not in richieste:                       # inizializzazione "a mano" dei 3 dizionari
            richieste[host] = 0
            errori[host] = 0
            latenze[host] = []
        richieste[host] += 1
        if status >= 500:
            errori[host] += 1
        latenze[host].append(rt)
    f.close()

# calcolo dei KPI e report: anch'essi dentro lo stesso flusso lineare
print("=== Report host (AS-IS) ===")
for host in richieste:
    n = richieste[host]
    err_rate = errori[host] / n
    ordinate = sorted(latenze[host])
    p95 = ordinate[int(len(ordinate) * 0.95)]           # indice del percentile calcolato "al volo"
    print(f"{host} | richieste: {n} | error_rate: {round(err_rate, 3)} | p95_ms: {p95}")
    if err_rate > 0.05:                                 # numeri magici: 0.05 e 800
        print("   ATTENZIONE: error rate alto")
    if p95 > 800:
        print("   ATTENZIONE: latenza p95 alta")


=== Report host (AS-IS) ===
web02 | richieste: 200 | error_rate: 0.09 | p95_ms: 1986
   ATTENZIONE: error rate alto
   ATTENZIONE: latenza p95 alta
web03 | richieste: 200 | error_rate: 0.03 | p95_ms: 108
web01 | richieste: 200 | error_rate: 0.01 | p95_ms: 90


**Funziona** — e su `web02` scattano correttamente gli allarmi. Ma guardiamolo con gli
occhi di chi lo dovrà mantenere e far girare in produzione:

- **Percorso cablato.** Per cambiare cartella si edita il codice. Non è riusabile su un altro
  ambiente senza modificarlo.
- **Tutto intrecciato.** Lettura, parsing, validazione, aggregazione, calcolo del percentile,
  report e *alerting* vivono nello stesso ciclo. Non puoi toccare un pezzo senza rischiare
  gli altri.
- **Dizionari paralleli.** `richieste`, `errori`, `latenze` vanno tenuti allineati a mano: una
  modifica e si disallineano.
- **Numeri magici.** `0.05`, `800`, `0.95`, `500` non hanno nome né un punto unico dove
  cambiarli.
- **Nessun riuso, nessun test.** Il calcolo è inseparabile dall'I/O: non puoi verificarlo in
  isolamento. E niente è importabile da un altro script.
- **Nessuna difesa sui dati.** Assume che *ogni* riga sia perfetta. In test lo è. In
  produzione, no — come vediamo subito qui sotto.

In [ ]:
# Cosa fa il parser ingenuo di fronte alle righe "vere"? Lo proviamo riga per riga,
# protetti da un try/except solo per non interrompere il notebook.
righe_problematiche = [
    "2025-06-20T15:01:10 host=web02 status=500",            # manca rt_ms
    "2025-06-20T15:01:11 host=web02 status=200 rt_ms=NaN",  # rt_ms non numerico
    "QUESTA RIGA E' ROTTA",                                  # niente chiave=valore
]

for riga in righe_problematiche:
    print(f"\nRiga: {riga!r}")
    try:
        parti = riga.split()
        host = status = rt = None
        for token in parti[1:]:
            chiave, valore = token.split("=")              # <-- esplode se manca '='
            if chiave == "host":   host = valore
            elif chiave == "status": status = int(valore)
            elif chiave == "rt_ms":  rt = int(valore)      # <-- esplode se non numerico
        # se siamo arrivati qui senza eccezioni, controlliamo cosa abbiamo ottenuto
        print(f"  risultato: host={host} status={status} rt={rt}")
        if rt is None:
            print("  >>> nessun errore qui, ma rt=None: un dato SBAGLIATO in silenzio,")
            print("      che farà detonare il calcolo del p95 a valle (sorted con None).")
    except Exception as e:
        print(f"  >>> lo script si INTERROMPE: {type(e).__name__}: {e}")



Riga: '2025-06-20T15:01:10 host=web02 status=500'
  risultato: host=web02 status=500 rt=None
  >>> nessun errore qui, ma rt=None: un dato SBAGLIATO in silenzio,
      che farà detonare il calcolo del p95 a valle (sorted con None).

Riga: '2025-06-20T15:01:11 host=web02 status=200 rt_ms=NaN'
  >>> lo script si INTERROMPE: ValueError: invalid literal for int() with base 10: 'NaN'

Riga: "QUESTA RIGA E' ROTTA"
  >>> lo script si INTERROMPE: ValueError: not enough values to unpack (expected 2, got 1)


Tre righe, **tre comportamenti diversi**: una passa producendo un dato sbagliato *in
silenzio*, due fanno saltare lo script. Su `logs_reali/` lo script as-is si interromperebbe
alla prima riga sporca — di notte, dentro a cron, senza che nessuno se ne accorga finché non
manca il report.

## 2. Verso lo strumento

Ricostruiamo la stessa attività separando le responsabilità, **un concern alla volta**. Ogni
pezzo avrà un solo compito e un nome; la logica starà in funzioni **pure**; l'I/O ai **bordi**;
gli errori saranno gestiti; le soglie diventeranno **configurazione**. Alla fine i pezzi si
comporranno in un unico strumento — con CLI, gestione errori, *exit code* e test.

### 2.1 Modellare i dati: `dataclass` + type hints

Primo passo: dare un **nome e una forma** ai dati, al posto dei dizionari paralleli.

- `Evento` rappresenta **una** richiesta letta dal log.
- `StatoHost` rappresenta i **KPI aggregati** di un host (con `error_rate` calcolato come
  proprietà: non è un dato da tenere allineato, è una funzione dei dati).
- `Soglie` raccoglie i **numeri magici** in un unico oggetto immutabile (`frozen=True`): un solo
  posto dove leggerli e cambiarli.

I **type hints** (`host: str`, `-> float`) documentano le intenzioni e permettono agli strumenti
(editor, linter, `mypy`) di trovare errori prima dell'esecuzione.

In [ ]:
import math
import logging
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

# Un logger configurato una volta sola. logging (non print) separa la DIAGNOSTICA
# (warning, info) dall'OUTPUT vero, e permette di filtrare per livello o dirottare su file.
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)
logger = logging.getLogger("monitor")


@dataclass
class Evento:
    """Una singola richiesta letta da una riga di log."""
    host: str
    status: int
    rt_ms: int


@dataclass
class StatoHost:
    """KPI aggregati di un host."""
    host: str
    n_richieste: int
    n_errori: int
    p95_ms: int

    @property
    def error_rate(self) -> float:
        # derivato dai dati: niente da tenere "allineato a mano"
        return self.n_errori / self.n_richieste if self.n_richieste else 0.0


@dataclass(frozen=True)
class Soglie:
    """Le soglie di allarme, in un unico posto e immutabili."""
    error_rate_max: float = 0.05   # oltre questa quota di 5xx => critico
    p95_ms_max: int = 800          # oltre questa latenza p95 (ms) => critico
    percentile: float = 0.95       # quale percentile usare per la latenza


### 2.2 Il parsing come funzione pura

Isoliamo la trasformazione *"riga di testo → `Evento`"* in una funzione **pura**: nessun I/O,
nessun logging, nessuno stato globale. Restituisce un `Evento` se la riga è valida, `None` se
non lo è. Essendo pura, è **banale da testare** (lo faremo) e **difensiva** per costruzione —
non assume nulla sull'input:

- `partition("=")` invece di `split("=")`: non esplode se manca o c'è più di un `=`;
- controlla che **tutti** i campi attesi siano presenti;
- intercetta i valori **non numerici** con `try/except`.

In [ ]:
def parse_riga(riga: str) -> Evento | None:
    """Trasforma una riga di log in un Evento, o restituisce None se non interpretabile.

    Funzione PURA: nessun I/O, nessun logging. Tutta la fragilità del mondo reale
    (campi mancanti, valori non numerici, righe spazzatura) è gestita qui, in un
    punto solo, ben definito e testabile.
    """
    campi: dict[str, str] = {}
    for token in riga.split()[1:]:          # salta il token 0 (il timestamp)
        if "=" not in token:
            return None                      # token che non è una coppia chiave=valore
        chiave, _, valore = token.partition("=")   # robusto: niente unpack che esplode
        campi[chiave] = valore

    # devono esserci TUTTI e tre i campi, altrimenti la riga non è utilizzabile
    if not {"host", "status", "rt_ms"}.issubset(campi):
        return None

    try:
        return Evento(
            host=campi["host"],
            status=int(campi["status"]),
            rt_ms=int(campi["rt_ms"]),
        )
    except ValueError:
        return None                          # status o rt_ms non numerici


### 2.3 L'I/O ai bordi: lettura robusta

Ora la lettura dei file — l'**I/O**, confinato in una funzione sua. Qui concentriamo tutto ciò
che riguarda *come* arrivano i dati, lasciando il resto del programma indifferente alla loro
provenienza:

- `with ... open(...)` (**context manager**): il file si chiude da solo, anche in caso di errore;
- **encoding esplicito**: niente sorprese tra ambienti;
- `cartella.glob("*.log")`: leggiamo **solo** i file di log — il `README.txt` estraneo è ignorato
  per costruzione, non per fortuna;
- iteriamo il file **riga per riga** (è un generatore): un log da gigabyte non viene caricato
  tutto in memoria — cruciale sui log ruotati della vostra infrastruttura;
- le righe invalide vengono **contate e segnalate** con `logging`, non fanno saltare nulla.

In [ ]:
def leggi_eventi(cartella: Path) -> list[Evento]:
    """Legge tutti i file *.log della cartella e restituisce gli Eventi validi.

    L'I/O sta tutto qui. Le righe non interpretabili vengono contate e segnalate
    via logging, ma NON interrompono l'elaborazione.
    """
    eventi: list[Evento] = []
    scartate = 0

    for percorso in sorted(cartella.glob("*.log")):     # solo i .log: il README viene ignorato
        with percorso.open(encoding="utf-8") as f:       # context manager + encoding esplicito
            for numero, riga in enumerate(f, start=1):   # streaming: una riga alla volta
                riga = riga.strip()
                if not riga:
                    continue                              # salta le righe vuote
                evento = parse_riga(riga)                 # la logica pura di parsing
                if evento is None:
                    scartate += 1
                    logger.warning("%s:%d riga non valida, saltata: %r",
                                   percorso.name, numero, riga)
                    continue
                eventi.append(evento)

    if scartate:
        logger.info("Righe scartate in totale: %d", scartate)
    logger.info("Eventi validi letti: %d", len(eventi))
    return eventi


### 2.4 Il calcolo come funzione pura: aggregazione e KPI

Il cuore dell'analisi, di nuovo come funzioni **pure** — ricevono dati, restituiscono dati,
non leggono né stampano nulla:

- `percentile(...)` calcola il p-esimo percentile con il metodo *nearest-rank* (esplicito e
  prevedibile), al posto dell'indice calcolato "al volo" della versione as-is;
- `aggrega(...)` raggruppa gli eventi per host (con un `defaultdict`, niente inizializzazione a
  mano) e produce gli `StatoHost`;
- `diagnosi(...)` decide se un host è critico e **perché**, restituendo l'elenco dei motivi
  (lista vuota = host sano).

Separare il *calcolo* dal *giudizio* (`diagnosi`) dal *report* significa che ognuno si può
cambiare e testare per conto suo.

In [ ]:
def percentile(valori: list[int], frazione: float) -> int:
    """p-esimo percentile (metodo nearest-rank) su una lista NON vuota."""
    ordinati = sorted(valori)
    indice = max(0, math.ceil(frazione * len(ordinati)) - 1)
    return ordinati[indice]


def aggrega(eventi: list[Evento], soglie: Soglie) -> list[StatoHost]:
    """Raggruppa gli eventi per host e calcola i KPI. Funzione PURA."""
    per_host: dict[str, list[Evento]] = defaultdict(list)   # niente init "a mano"
    for e in eventi:
        per_host[e.host].append(e)

    stati: list[StatoHost] = []
    for host, lista in sorted(per_host.items()):
        n_richieste = len(lista)
        n_errori = sum(1 for e in lista if e.status >= 500)
        p95 = percentile([e.rt_ms for e in lista], soglie.percentile)
        stati.append(StatoHost(host=host, n_richieste=n_richieste,
                               n_errori=n_errori, p95_ms=p95))
    return stati


def diagnosi(stato: StatoHost, soglie: Soglie) -> list[str]:
    """Elenca i motivi per cui un host è critico. Lista vuota = host sano. PURA."""
    motivi = []
    if stato.error_rate > soglie.error_rate_max:
        motivi.append(f"error_rate {stato.error_rate:.1%} > {soglie.error_rate_max:.1%}")
    if stato.p95_ms > soglie.p95_ms_max:
        motivi.append(f"p95 {stato.p95_ms}ms > {soglie.p95_ms_max}ms")
    return motivi


### 2.5 Il report (I/O) e l'orchestrazione

Il report stampa a video: è **I/O**, quindi sta in una funzione sua, separata dal calcolo.
`main` è l'**orchestratore**: mette in fila i passi (leggi → aggrega → riporta) senza contenere
logica di dettaglio. Si legge come un indice del programma.

In [ ]:
def stampa_report(stati: list[StatoHost], soglie: Soglie) -> None:
    """Stampa il report a video. È I/O: nessun calcolo qui dentro."""
    print("=== Report host ===")
    for s in stati:
        motivi = diagnosi(s, soglie)
        esito = "OK" if not motivi else "CRITICO"
        print(f"{s.host:<8} | rich: {s.n_richieste:>4} | "
              f"err: {s.error_rate:>6.1%} | p95: {s.p95_ms:>5}ms | {esito}")
        for m in motivi:
            print(f"         -> {m}")


def main(cartella: Path, soglie: Soglie = Soglie()) -> list[StatoHost]:
    """Orchestratore: legge, aggrega, riporta. Restituisce gli stati (utile per i test)."""
    eventi = leggi_eventi(cartella)        # I/O ai bordi
    stati = aggrega(eventi, soglie)        # logica pura
    stampa_report(stati, soglie)           # I/O ai bordi
    return stati


Eseguiamo lo strumento sui **log reali** (`logs_reali/`, quelli con le righe sporche e il
file estraneo). Le righe invalide compaiono come `WARNING` di `logging` — **separate** dal
report — e l'elaborazione arriva in fondo. Dove l'as-is si interrompeva, lo strumento
**sopravvive e ti dice cosa ha scartato**.

In [ ]:
stati = main(Path("logs_reali"))


### 2.6 Da funzione a strumento: CLI, *exit code*, punto di ingresso

L'ultimo passo che trasforma il codice in uno **strumento da terminale**:

- i percorsi e le soglie diventano **parametri** via `argparse` — niente più valori cablati;
- un **exit code** (0 = tutto OK, 1 = almeno un host critico) permette a cron/scheduler di
  capire l'esito *senza leggere l'output*: è così che un tool si integra in una pipeline;
- il blocco `if __name__ == "__main__"` rende il file **eseguibile** da riga di comando **e**
  **importabile** come modulo (per i test, o per riusarne le funzioni altrove).

In uno script reale lo lanciereste così:

```bash
python monitor.py --cartella /var/log/app --error-rate-max 0.03
echo $?   # 0 oppure 1
```

Qui nel notebook simuliamo la riga di comando passando una lista esplicita a `parse_args`.

In [ ]:
import argparse
import sys


def parse_args(argv=None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Analizza i log applicativi e segnala gli host critici.")
    parser.add_argument("--cartella", type=Path, default=Path("logs_reali"),
                        help="Cartella contenente i file *.log")
    parser.add_argument("--error-rate-max", type=float, default=0.05,
                        help="Soglia di error rate oltre la quale l'host è critico")
    parser.add_argument("--p95-ms-max", type=int, default=800,
                        help="Soglia di latenza p95 (ms) oltre la quale l'host è critico")
    return parser.parse_args(argv)


def cli(argv=None) -> int:
    """Punto di ingresso: traduce gli argomenti, esegue, restituisce l'exit code."""
    args = parse_args(argv)
    soglie = Soglie(error_rate_max=args.error_rate_max, p95_ms_max=args.p95_ms_max)
    stati = main(args.cartella, soglie)
    critici = [s for s in stati if diagnosi(s, soglie)]
    return 1 if critici else 0          # 1 se c'è almeno un host critico, altrimenti 0


# In uno script reale, l'unica riga "a livello di modulo":
#
#     if __name__ == "__main__":
#         sys.exit(cli())
#
# Simulazione della riga di comando dentro al notebook:
codice_uscita = cli(["--cartella", "logs_reali", "--error-rate-max", "0.05"])
print(f"\nExit code: {codice_uscita}  (0 = tutto OK, 1 = almeno un host critico)")


WARNING web02.log:201 riga non valida, saltata: '2025-06-20T15:01:10 host=web02 status=500'
WARNING web02.log:202 riga non valida, saltata: '2025-06-20T15:01:11 host=web02 status=200 rt_ms=NaN'
WARNING web02.log:203 riga non valida, saltata: "QUESTA RIGA E' ROTTA"
WARNING web02.log:204 riga non valida, saltata: '2025-06-20T15:01:13 status=200 rt_ms=55'
INFO Righe scartate in totale: 4
INFO Eventi validi letti: 600


=== Report host ===
web01    | rich:  200 | err:   1.0% | p95:    89ms | OK
web02    | rich:  200 | err:   9.0% | p95:  1932ms | CRITICO
         -> error_rate 9.0% > 5.0%
         -> p95 1932ms > 800ms
web03    | rich:  200 | err:   3.0% | p95:   108ms | OK

Exit code: 1  (0 = tutto OK, 1 = almeno un host critico)


### 2.7 Un primo assaggio di test

Ecco il pagamento del lavoro fatto: avendo isolato la logica in **funzioni pure**, testarla è
immediato — input noto, output atteso, nessun file o sistema da predisporre. È *impossibile*
farlo sulla versione as-is, dove il calcolo è inchiodato all'I/O.

In un progetto vero questi test stanno in un file `test_monitor.py` e si lanciano con:

```bash
pytest -v
```

`pytest` raccoglie da solo tutte le funzioni `test_*` e segnala quali passano. Qui — per restare
autosufficienti — definiamo gli stessi test e li eseguiamo con un mini-runner inline, ma la
**forma** è esattamente quella di pytest.

In [ ]:
def test_parse_riga_valida():
    e = parse_riga("2025-06-20T14:00:01 host=web01 status=200 rt_ms=42")
    assert e == Evento(host="web01", status=200, rt_ms=42)

def test_parse_riga_non_numerica():
    assert parse_riga("2025-06-20T14:00:01 host=web01 status=200 rt_ms=NaN") is None

def test_parse_riga_campo_mancante():
    assert parse_riga("2025-06-20T14:00:01 host=web01 status=200") is None      # manca rt_ms

def test_parse_riga_spazzatura():
    assert parse_riga("QUESTA RIGA E' ROTTA") is None

def test_percentile_nearest_rank():
    assert percentile([10, 20, 30, 40, 50], 0.95) == 50
    assert percentile([42], 0.95) == 42

def test_aggrega_kpi():
    eventi = [
        Evento("a", 200, 10),
        Evento("a", 500, 20),   # 1 errore su 4
        Evento("a", 200, 30),
        Evento("a", 200, 40),
    ]
    [stato] = aggrega(eventi, Soglie())
    assert stato.n_richieste == 4
    assert stato.n_errori == 1
    assert stato.error_rate == 0.25


def esegui_test(namespace) -> None:
    """Mini-runner: esegue tutte le funzioni test_* del namespace (al posto di pytest)."""
    falliti = 0
    for nome, fn in sorted(namespace.items()):
        if nome.startswith("test_") and callable(fn):
            try:
                fn()
                print(f"PASS  {nome}")
            except AssertionError as e:
                falliti += 1
                print(f"FAIL  {nome}: {e}")
    print("\n" + ("TUTTI I TEST PASSANO" if not falliti else f"{falliti} TEST FALLITI"))


esegui_test(dict(globals()))


PASS  test_aggrega_kpi
PASS  test_parse_riga_campo_mancante
PASS  test_parse_riga_non_numerica
PASS  test_parse_riga_spazzatura
PASS  test_parse_riga_valida
PASS  test_percentile_nearest_rank

TUTTI I TEST PASSANO


## Confronto: as-is → strumento

| Aspetto | **AS-IS** (script lineare) | **Strumento** |
|---|---|---|
| Organizzazione | un unico blocco | funzioni a responsabilità singola |
| Percorsi e soglie | cablati nel codice (numeri magici) | parametri + `Soglie` (un posto solo) |
| Modello dati | dizionari paralleli da allineare | `dataclass` (`Evento`, `StatoHost`) |
| Parsing | mescolato, assume input perfetto | funzione **pura** e difensiva |
| Lettura file | `open` senza `with`/encoding, legge tutto | `with` + encoding + `glob` + streaming |
| Calcolo | accoppiato all'I/O | funzioni **pure** (`aggrega`, `percentile`) |
| Righe sporche | lo script si interrompe | contate, segnalate, saltate |
| Diagnostica | `print` mescolato all'output | `logging` separato per livello |
| Integrazione | nessun esito programmatico | **exit code** per cron/scheduler |
| Test | impossibile | banale sulle funzioni pure |

## Cosa abbiamo guadagnato (e quando fermarsi)

Non abbiamo aggiunto funzionalità: l'output è lo stesso. Abbiamo cambiato **come** è fatto, e
con questo abbiamo ottenuto uno strumento che **sopravvive ai dati reali**, **dice cosa è andato
storto**, si **integra** con uno scheduler, si **modifica senza paura** (i test lo proteggono) e
si **riusa** come modulo.

Questo **non** significa che ogni script vada ristrutturato così: per un'estrazione *una tantum*
da lanciare a mano, la forma lineare va benissimo — la struttura è un investimento, e ha senso
quando il codice viene **rieseguito**, **cresce**, va **condiviso** o **testato**. Il punto del
modulo è saper riconoscere *quando* fare il salto e *come*.

Lo scheletro costruito qui — **leggi (I/O) → trasforma (puro) → aggrega (puro) → riporta (I/O)**,
con configurazione, logging, gestione errori e test — è la **spina dorsale** di tutto il corso.
Nei prossimi giorni le sorgenti diventeranno Oracle, SQL Server e Splunk, e la lettura verrà
**parallelizzata** su centinaia di host: ma la struttura resterà questa.